In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

from lifelines import CoxPHFitter
from lifelines.utils import concordance_index
from scipy.stats import chi2

In [30]:
# load data
DATA_PATH = "/mnt/DataVol/Beratungen/Yurttas/survival/data"

df = pd.read_csv(f"{DATA_PATH}/GPT_processed_survival_data.csv")

In [31]:
df.head()

,Bday,OPDate,Age,Tumor,sPCI,pPCI,Tod Datum,Datum Rezidiv,time,months,...,Age_final,Tumor_grouped_2,Tumor_grouped_3,Tumor_grouped_4,Tumor_grouped_5,Tumor_grouped_6,Tumor_grouped_7,Tumor_grouped_8,Tumor_grouped_Other,Sex_1
0,1991-02-25,2008-03-10,17,1,26,22,2008-08-16,2008-08-12,159.0,5.223390,...,17,False,False,False,False,False,False,False,False,True
1,1993-07-17,2015-07-14,21,1,5,3,2018-05-06,2016-12-22,1027.0,33.738502,...,21,False,False,False,False,False,False,False,False,False
2,1995-06-08,2018-11-23,23,4,15,6,NaN,2022-06-22,1307.0,42.936925,...,23,False,False,True,False,False,False,False,False,True
3,1979-09-02,2005-11-15,26,10,15,6,2009-05-11,2006-04-12,1273.0,41.819974,...,26,False,False,False,False,False,False,False,True,False
4,1990-11-18,2017-12-29,27,3,18,4,2019-11-03,2018-07-19,674.0,22.141919,...,27,False,True,False,False,False,False,False,False,False


In [32]:
cols_to_drop = [
    "Bday", "OPDate", "Tod Datum", "Datum Rezidiv", "Age_calc", "Age", "Tumor", "time", "Sex_1"
]
df = df.drop(columns=cols_to_drop, inplace=False)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 285 entries, 0 to 284
Data columns (total 13 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   sPCI                 285 non-null    int64  
 1   pPCI                 285 non-null    int64  
 2   months               285 non-null    float64
 3   event                285 non-null    int64  
 4   Age_final            285 non-null    int64  
 5   Tumor_grouped_2      285 non-null    bool   
 6   Tumor_grouped_3      285 non-null    bool   
 7   Tumor_grouped_4      285 non-null    bool   
 8   Tumor_grouped_5      285 non-null    bool   
 9   Tumor_grouped_6      285 non-null    bool   
 10  Tumor_grouped_7      285 non-null    bool   
 11  Tumor_grouped_8      285 non-null    bool   
 12  Tumor_grouped_Other  285 non-null    bool   
dtypes: bool(8), float64(1), int64(4)
memory usage: 13.5 KB


In [33]:
# base model
cph_base = CoxPHFitter()
cph_base.fit(
    df.drop(columns=["pPCI", "sPCI"]),
    duration_col="months",
    event_col="event"
)
cph_base.print_summary()

<lifelines.CoxPHFitter: fitted with 285 total observations, 154 right-censored observations>
             duration col = 'months'
                event col = 'event'
      baseline estimation = breslow
   number of observations = 285
number of events observed = 131
   partial log-likelihood = -528.71
         time fit was run = 2026-04-09 10:37:03 UTC

---
                     coef exp(coef)  se(coef)  coef lower 95%  coef upper 95% exp(coef) lower 95% exp(coef) upper 95%
covariate                                                                                                            
Age_final           -0.01      0.99      0.01           -0.03            0.01                0.97                1.01
Tumor_grouped_2     -0.45      0.64      0.38           -1.20            0.30                0.30                1.36
Tumor_grouped_3     -0.80      0.45      0.34           -1.46           -0.14                0.23                0.87
Tumor_grouped_4     -1.44      0.24      0.31           -2.05           -0.84                0.13                0.43
Tumor_grouped_5     -1.21      0.30      0.47           -2.13           -0.29                0.12                0.75
Tumor_grouped_6      0.59      1.80      0.29            0.03            1.15                1.03                3.15
Tumor_grouped_7     -0.84      0.43      0.47           -1.77            0.08                0.17                1.08
Tumor_grouped_8      0.47      1.59      0.48           -0.47            1.40                0.63                4.05
Tumor_grouped_Other -0.17      0.84      0.37           -0.89            0.55                0.41                1.74

                     cmp to     z      p  -log2(p)
covariate                                         
Age_final              0.00 -1.18   0.24      2.08
Tumor_grouped_2        0.00 -1.17   0.24      2.04
Tumor_grouped_3        0.00 -2.38   0.02      5.84
Tumor_grouped_4        0.00 -4.69 <0.005     18.50
Tumor_grouped_5        0.00 -2.58   0.01      6.64
Tumor_grouped_6        0.00  2.07   0.04      4.70
Tumor_grouped_7        0.00 -1.79   0.07      3.77
Tumor_grouped_8        0.00  0.98   0.33      1.61
Tumor_grouped_Other    0.00 -0.46   0.64      0.63
---
Concordance = 0.67
Partial AIC = 1075.41
log-likelihood ratio test = 49.68 on 9 df
-log2(p) of ll-ratio test = 22.95

In [34]:
# model A: sPCI
cph_sPCI = CoxPHFitter()
cph_sPCI.fit(
    df.drop(columns=["pPCI"]),
    duration_col="months",
    event_col="event"
)
cph_sPCI.print_summary()

<lifelines.CoxPHFitter: fitted with 285 total observations, 154 right-censored observations>
             duration col = 'months'
                event col = 'event'
      baseline estimation = breslow
   number of observations = 285
number of events observed = 131
   partial log-likelihood = -526.44
         time fit was run = 2026-04-09 10:37:03 UTC

---
                     coef exp(coef)  se(coef)  coef lower 95%  coef upper 95% exp(coef) lower 95% exp(coef) upper 95%
covariate                                                                                                            
sPCI                 0.03      1.03      0.01            0.00            0.05                1.00                1.05
Age_final           -0.01      0.99      0.01           -0.03            0.01                0.98                1.01
Tumor_grouped_2     -0.25      0.78      0.40           -1.02            0.53                0.36                1.70
Tumor_grouped_3     -1.00      0.37      0.35           -1.69           -0.31                0.18                0.73
Tumor_grouped_4     -1.40      0.25      0.31           -2.00           -0.80                0.13                0.45
Tumor_grouped_5     -1.45      0.23      0.48           -2.40           -0.50                0.09                0.61
Tumor_grouped_6      0.75      2.12      0.30            0.17            1.33                1.18                3.78
Tumor_grouped_7     -1.22      0.29      0.51           -2.22           -0.23                0.11                0.79
Tumor_grouped_8      0.48      1.62      0.48           -0.45            1.41                0.64                4.12
Tumor_grouped_Other -0.20      0.82      0.37           -0.92            0.52                0.40                1.69

                     cmp to     z      p  -log2(p)
covariate                                         
sPCI                   0.00  2.14   0.03      4.94
Age_final              0.00 -1.02   0.31      1.70
Tumor_grouped_2        0.00 -0.63   0.53      0.91
Tumor_grouped_3        0.00 -2.83 <0.005      7.77
Tumor_grouped_4        0.00 -4.54 <0.005     17.47
Tumor_grouped_5        0.00 -3.00 <0.005      8.52
Tumor_grouped_6        0.00  2.53   0.01      6.46
Tumor_grouped_7        0.00 -2.42   0.02      6.01
Tumor_grouped_8        0.00  1.01   0.31      1.68
Tumor_grouped_Other    0.00 -0.54   0.59      0.77
---
Concordance = 0.69
Partial AIC = 1072.89
log-likelihood ratio test = 54.21 on 10 df
-log2(p) of ll-ratio test = 24.43

In [35]:
# model B: pPCI
cph_pPCI = CoxPHFitter()
cph_pPCI.fit(
    df.drop(columns=["sPCI"]),
    duration_col="months",
    event_col="event"
)
cph_pPCI.print_summary()

<lifelines.CoxPHFitter: fitted with 285 total observations, 154 right-censored observations>
             duration col = 'months'
                event col = 'event'
      baseline estimation = breslow
   number of observations = 285
number of events observed = 131
   partial log-likelihood = -524.88
         time fit was run = 2026-04-09 10:37:03 UTC

---
                     coef exp(coef)  se(coef)  coef lower 95%  coef upper 95% exp(coef) lower 95% exp(coef) upper 95%
covariate                                                                                                            
pPCI                 0.05      1.05      0.02            0.01            0.08                1.01                1.09
Age_final           -0.01      0.99      0.01           -0.03            0.01                0.97                1.01
Tumor_grouped_2     -0.21      0.81      0.39           -0.99            0.56                0.37                1.75
Tumor_grouped_3     -0.94      0.39      0.34           -1.61           -0.27                0.20                0.77
Tumor_grouped_4     -1.43      0.24      0.31           -2.04           -0.82                0.13                0.44
Tumor_grouped_5     -1.38      0.25      0.48           -2.31           -0.44                0.10                0.64
Tumor_grouped_6      0.84      2.31      0.30            0.24            1.43                1.28                4.17
Tumor_grouped_7     -0.96      0.38      0.47           -1.89           -0.04                0.15                0.96
Tumor_grouped_8      0.41      1.51      0.48           -0.52            1.35                0.59                3.84
Tumor_grouped_Other -0.25      0.78      0.37           -0.97            0.47                0.38                1.61

                     cmp to     z      p  -log2(p)
covariate                                         
pPCI                   0.00  2.78   0.01      7.52
Age_final              0.00 -1.30   0.19      2.38
Tumor_grouped_2        0.00 -0.54   0.59      0.76
Tumor_grouped_3        0.00 -2.74   0.01      7.35
Tumor_grouped_4        0.00 -4.60 <0.005     17.88
Tumor_grouped_5        0.00 -2.89 <0.005      8.00
Tumor_grouped_6        0.00  2.77   0.01      7.47
Tumor_grouped_7        0.00 -2.04   0.04      4.58
Tumor_grouped_8        0.00  0.87   0.39      1.37
Tumor_grouped_Other    0.00 -0.68   0.50      1.00
---
Concordance = 0.70
Partial AIC = 1069.76
log-likelihood ratio test = 57.33 on 10 df
-log2(p) of ll-ratio test = 26.37

In [36]:
# model C: sPCI + pPCI
cph_both = CoxPHFitter()
cph_both.fit(
    df,
    duration_col="months",
    event_col="event"
)
cph_both.print_summary()

<lifelines.CoxPHFitter: fitted with 285 total observations, 154 right-censored observations>
             duration col = 'months'
                event col = 'event'
      baseline estimation = breslow
   number of observations = 285
number of events observed = 131
   partial log-likelihood = -524.83
         time fit was run = 2026-04-09 10:37:03 UTC

---
                     coef exp(coef)  se(coef)  coef lower 95%  coef upper 95% exp(coef) lower 95% exp(coef) upper 95%
covariate                                                                                                            
sPCI                 0.01      1.01      0.02           -0.03            0.04                0.97                1.04
pPCI                 0.04      1.05      0.02           -0.00            0.09                1.00                1.10
Age_final           -0.01      0.99      0.01           -0.03            0.01                0.97                1.01
Tumor_grouped_2     -0.20      0.82      0.40           -0.98            0.58                0.38                1.79
Tumor_grouped_3     -0.97      0.38      0.36           -1.67           -0.27                0.19                0.76
Tumor_grouped_4     -1.42      0.24      0.31           -2.03           -0.81                0.13                0.44
Tumor_grouped_5     -1.40      0.25      0.48           -2.35           -0.46                0.10                0.63
Tumor_grouped_6      0.84      2.32      0.30            0.25            1.44                1.28                4.21
Tumor_grouped_7     -1.03      0.36      0.51           -2.04           -0.02                0.13                0.98
Tumor_grouped_8      0.42      1.53      0.48           -0.51            1.36                0.60                3.89
Tumor_grouped_Other -0.25      0.78      0.37           -0.97            0.48                0.38                1.61

                     cmp to     z      p  -log2(p)
covariate                                         
sPCI                   0.00  0.33   0.74      0.43
pPCI                   0.00  1.78   0.07      3.74
Age_final              0.00 -1.24   0.21      2.22
Tumor_grouped_2        0.00 -0.50   0.62      0.69
Tumor_grouped_3        0.00 -2.73   0.01      7.29
Tumor_grouped_4        0.00 -4.57 <0.005     17.66
Tumor_grouped_5        0.00 -2.90 <0.005      8.08
Tumor_grouped_6        0.00  2.78   0.01      7.55
Tumor_grouped_7        0.00 -2.00   0.05      4.46
Tumor_grouped_8        0.00  0.89   0.38      1.41
Tumor_grouped_Other    0.00 -0.67   0.50      1.00
---
Concordance = 0.70
Partial AIC = 1071.66
log-likelihood ratio test = 57.44 on 11 df
-log2(p) of ll-ratio test = 25.11

In [37]:
print("C-index (base):", cph_base.concordance_index_)
print("C-index (sPCI):", cph_sPCI.concordance_index_)
print("C-index (pPCI):", cph_pPCI.concordance_index_)
print("C-index (both):", cph_both.concordance_index_)

C-index (base): 0.6715187523208318
C-index (sPCI): 0.689528406981062
C-index (pPCI): 0.6990345339769773
C-index (both): 0.6998514667656888


In [38]:
ll_A = cph_sPCI.log_likelihood_
ll_C = cph_both.log_likelihood_

LR_stat = 2 * (ll_C - ll_A)
df_diff = len(cph_both.params_) - len(cph_sPCI.params_)

p_value = chi2.sf(LR_stat, df_diff)

print("LRT p-value (pPCI adds beyond sPCI):", p_value)

LRT p-value (pPCI adds beyond sPCI): 0.07233141900324587


In [39]:
cph_both.check_assumptions(df, p_value_threshold=0.05)

The ``p_value_threshold`` is set at 0.05. Even under the null hypothesis of no violations, some
covariates will be below the threshold by chance. This is compounded when there are many covariates.
Similarly, when there are lots of observations, even minor deviances from the proportional hazard
assumption will be flagged.

With that in mind, it's best to use a combination of statistical tests and visual tests to determine
the most serious violations. Produce visual plots using ``check_assumptions(..., show_plots=True)``
and looking for non-constant lines. See link [A] below for a full example.





1. Variable 'Tumor_grouped_Other' failed the non-proportional test: p-value is 0.0284.

   Advice: with so few unique values (only 2), you can include `strata=['Tumor_grouped_Other', ...]`
in the call in `.fit`. See documentation in link [E] below.

---
[A]  https://lifelines.readthedocs.io/en/latest/jupyter_notebooks/Proportional%20hazard%20assumption.html
[B]  https://lifelines.readthedocs.io/en/latest/jupyter_notebooks/Proportional%20hazard%20assumption.html#Bin-variable-and-stratify-on-it
[C]  https://lifelines.readthedocs.io/en/latest/jupyter_notebooks/Proportional%20hazard%20assumption.html#Introduce-time-varying-covariates
[D]  https://lifelines.readthedocs.io/en/latest/jupyter_notebooks/Proportional%20hazard%20assumption.html#Modify-the-functional-form
[E]  https://lifelines.readthedocs.io/en/latest/jupyter_notebooks/Proportional%20hazard%20assumption.html#Stratification



[]

In [ ]:
df_=